<h2 align='center'>ML Flow</h2>

In [1]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report
import warnings
warnings.filterwarnings('ignore')

In [2]:
# Step 1: Create an imbalanced binary classification dataset
X, y = make_classification(n_samples=1000, n_features=10, n_informative=2, n_redundant=8, 
                           weights=[0.9, 0.1], flip_y=0, random_state=42)

np.unique(y, return_counts=True)

(array([0, 1]), array([900, 100]))

In [3]:
# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)

#### Handle class imbalance

In [4]:
from imblearn.combine import SMOTETomek

smt = SMOTETomek(random_state=42)
X_train_res, y_train_res = smt.fit_resample(X_train, y_train)
np.unique(y_train_res, return_counts=True)

(array([0, 1]), array([619, 619]))

### Track Experiments

In [5]:
models = [
    (
        "Logistic Regression", 
        {"C": 1, "solver": 'liblinear'},
        LogisticRegression(), 
        (X_train, y_train),
        (X_test, y_test)
    ),
    (
        "Random Forest", 
        {"n_estimators": 30, "max_depth": 3},
        RandomForestClassifier(), 
        (X_train, y_train),
        (X_test, y_test)
    ),
    (
        "XGBClassifier",
        {"use_label_encoder": False, "eval_metric": 'logloss'},
        XGBClassifier(), 
        (X_train, y_train),
        (X_test, y_test)
    ),
    (
        "XGBClassifier With SMOTE",
        {"use_label_encoder": False, "eval_metric": 'logloss'},
        XGBClassifier(), 
        (X_train_res, y_train_res),
        (X_test, y_test)
    )
]

In [6]:
reports = []

for model_name, params, model, train_set, test_set in models:
    X_train = train_set[0]
    y_train = train_set[1]
    X_test = test_set[0]
    y_test = test_set[1]
    
    model.set_params(**params)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    report = classification_report(y_test, y_pred, output_dict=True)
    reports.append(report)

In [7]:
import mlflow
import mlflow.sklearn
import mlflow.xgboost

In [8]:
# Initialize MLflow
mlflow.set_experiment("Anomaly Detection")
mlflow.set_tracking_uri("http://127.0.0.1:5000")

for i, element in enumerate(models):
    model_name = element[0]
    params = element[1]
    model = element[2]
    report = reports[i]
    
    with mlflow.start_run(run_name=model_name):        
        mlflow.log_params(params)
        mlflow.log_metrics({
            'accuracy': report['accuracy'],
            'recall_class_1': report['1']['recall'],
            'recall_class_0': report['0']['recall'],
            'f1_score_macro': report['macro avg']['f1-score']
        })  
        
        if "XGB" in model_name:
            mlflow.xgboost.log_model(model, "model")
        else:
            mlflow.sklearn.log_model(model, "model")  

2025/04/23 17:32:49 INFO mlflow.tracking.fluent: Experiment with name 'Anomaly Detection' does not exist. Creating a new experiment.
2025/04/23 17:32:53 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


🏃 View run Logistic Regression at: http://127.0.0.1:5000/#/experiments/663886932846066599/runs/ff22fc15378c45d9a908b16fb6907d4f
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/663886932846066599


2025/04/23 17:32:56 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


🏃 View run Random Forest at: http://127.0.0.1:5000/#/experiments/663886932846066599/runs/92da68b97ac24a1289602151d6ba6735
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/663886932846066599


2025/04/23 17:32:59 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


🏃 View run XGBClassifier at: http://127.0.0.1:5000/#/experiments/663886932846066599/runs/901146b1766f4c7eab242d16ebac6a33
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/663886932846066599


2025/04/23 17:33:02 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


🏃 View run XGBClassifier With SMOTE at: http://127.0.0.1:5000/#/experiments/663886932846066599/runs/54dab0591021445a85688d790ee0b330
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/663886932846066599


### Register the Model

In [9]:
model_name = 'XGB-Smote'
run_id=input('Please type RunID')
model_uri = f'runs:/{run_id}/model'

with mlflow.start_run(run_id=run_id):
    mlflow.register_model(model_uri=model_uri, name=model_name)

Successfully registered model 'XGB-Smote'.
2025/04/23 17:33:21 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: XGB-Smote, version 1


🏃 View run XGBClassifier With SMOTE at: http://127.0.0.1:5000/#/experiments/663886932846066599/runs/54dab0591021445a85688d790ee0b330
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/663886932846066599


Created version '1' of model 'XGB-Smote'.


In [22]:
print(model_uri)
# model_uri = models:/XGB-Smote/1
# actual path :E:\Projects\Ml_ops\ML_Assignment\MLOps_Deployment\mlruns\models\XGB-Smote\version-1

models:/XGB-Smote/1


### Load the Model

In [21]:
from mlflow.tracking import MlflowClient

client = MlflowClient()
model_info = client.get_model_version("XGB-Smote", "1")
print("Model Source:", model_info.source)

Model Source: file:///e:/Projects/Ml_ops/ML_Assignment/MLOps_Deployment/mlruns/663886932846066599/54dab0591021445a85688d790ee0b330/artifacts/model


In [12]:
import mlflow
runs = mlflow.search_runs()
print(runs[["run_id", "artifact_uri"]].to_string(index=False))


                          run_id                                                                                                                   artifact_uri
54dab0591021445a85688d790ee0b330 file:///e:/Projects/Ml_ops/ML_Assignment/MLOps_Deployment/mlruns/663886932846066599/54dab0591021445a85688d790ee0b330/artifacts
901146b1766f4c7eab242d16ebac6a33 file:///e:/Projects/Ml_ops/ML_Assignment/MLOps_Deployment/mlruns/663886932846066599/901146b1766f4c7eab242d16ebac6a33/artifacts
92da68b97ac24a1289602151d6ba6735 file:///e:/Projects/Ml_ops/ML_Assignment/MLOps_Deployment/mlruns/663886932846066599/92da68b97ac24a1289602151d6ba6735/artifacts
ff22fc15378c45d9a908b16fb6907d4f file:///e:/Projects/Ml_ops/ML_Assignment/MLOps_Deployment/mlruns/663886932846066599/ff22fc15378c45d9a908b16fb6907d4f/artifacts


In [13]:
model_version = 1
model_uri = f"models:/{model_name}/{model_version}"

loaded_model = mlflow.xgboost.load_model(model_uri)
y_pred = loaded_model.predict(X_test)
y_pred[:4]

array([0, 0, 0, 0])

### Transition the Model to Production

In [15]:
current_model_uri = f"models:/{model_name}@challenger"
production_model_name = "anomaly-detection-prod"

client = mlflow.MlflowClient()
client.copy_model_version(src_model_uri=current_model_uri, dst_name=production_model_name)

Successfully registered model 'anomaly-detection-prod'.
Copied version '1' of model 'XGB-Smote' to version '1' of model 'anomaly-detection-prod'.


<ModelVersion: aliases=[], creation_timestamp=1745409829882, current_stage='None', description='', last_updated_timestamp=1745409829882, name='anomaly-detection-prod', run_id='54dab0591021445a85688d790ee0b330', run_link='', source='models:/XGB-Smote/1', status='READY', status_message=None, tags={}, user_id='', version='1'>

In [17]:
model_version = 1
prod_model_uri = f"models:/{production_model_name}@champion"

loaded_model = mlflow.xgboost.load_model(prod_model_uri)
y_pred = loaded_model.predict(X_test)
y_pred[:50]

array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 1])

Please refer to following to learn more about model registry

https://mlflow.org/docs/latest/model-registry.html#model-registry-workflows to learn 

In [20]:
import requests
import json

url = "http://127.0.0.1:5001/invocations"
headers = {"Content-Type": "application/json"}

# Replace with actual column names and 2D array of data
data = {
    "inputs": {
        "columns": ["feature1", "feature2", "feature3"],
        "data": [
            [1.2, 3.4, 5.6],
            [7.8, 9.0, 1.2]
        ]
    }
}

response = requests.post(url, headers=headers, data=json.dumps(data))
print("✅ Prediction Response:", response.json())

✅ Prediction Response: {'error_code': 'BAD_REQUEST', 'message': 'Encountered an unexpected error while evaluating the model. Verify that the serialized input Dataframe is compatible with the model for inference.', 'stack_trace': 'Traceback (most recent call last):\n  File "e:\\Projects\\Ml_ops\\.venv\\Lib\\site-packages\\mlflow\\pyfunc\\scoring_server\\__init__.py", line 369, in invocations\n    raw_predictions = model.predict(data, params=params)\n                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^\n  File "e:\\Projects\\Ml_ops\\.venv\\Lib\\site-packages\\mlflow\\pyfunc\\__init__.py", line 804, in predict\n    return self._predict(data, params)\n           ^^^^^^^^^^^^^^^^^^^^^^^^^^^\n  File "e:\\Projects\\Ml_ops\\.venv\\Lib\\site-packages\\mlflow\\pyfunc\\__init__.py", line 854, in _predict\n    return self._predict_fn(data, params=params)\n           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^\n  File "e:\\Projects\\Ml_ops\\.venv\\Lib\\site-packages\\mlflow\\xgboost\\__init__.py", li

In [24]:
# Load the model from the registry
model_uri = "models:/XGB-Smote/1"
model = mlflow.pyfunc.load_model(model_uri)

# Check if the model has a `columns` or `feature_names_` attribute
# This works for models like XGBoost which store feature names

if hasattr(model, 'feature_names_in_'):
    feature_names = model.feature_names_in_
    print(f"Feature names: {feature_names}")
else:
    print("Feature names not available in the model")


Feature names not available in the model


In [ ]:
# Check if X_train_res is a DataFrame
if isinstance(X_train_res, pd.DataFrame):
    feature_names = X_train_res.columns.tolist()
    print(f"Feature names: {feature_names}")
else:
    print("X_train_res is not a DataFrame, can't extract feature names automatically.")

NameError: name 'pd' is not defined

In [25]:
import xgboost as xgb

# Load the XGBoost model
model = xgb.Booster()
model.load_model('E:/Projects/Ml_ops/ML_Assignment/MLOps_Deployment/mlruns/663886932846066599/54dab0591021445a85688d790ee0b330/artifacts/model/model.xgb')  # Path to your .xgb file

# Get the feature names
feature_names = model.feature_names
print(f"Feature names: {feature_names}")


Feature names: None
